In [34]:
import sys
from pathlib import Path

ROOT = Path().resolve().parents[1] # go up n levels (adjust as needed)
sys.path.append(str(ROOT))

from config import PROJECT_ROOT, APT_ROOT
from apt_project import *

In [35]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import numpy as np

In [40]:
# Define feature columns
feature_cols = ['year', 'month', 'industry_code', 'event_type', 'event_subtype', 'motive', 'actor_type']

# Identify Categorical vs Numeric
categorical_cols = ['industry_code', 'event_type', 'event_subtype', 'motive', 'actor_type']
numeric_cols =['year', 'month']

In [41]:
# Preprocess with one-hot encoding (fit on all data)
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

# Fit preprocessor on full dataset
preprocessor.fit(events_df[feature_cols])

,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,categories,'auto'
,drop,None
,sparse_output,True


In [42]:
# Filter out NaN values for the respective training sets
train_df_origin = events_df[events_df['origin_risk_norm'].notna()]
train_df_victim = events_df[events_df['victim_risk_norm'].notna()]

In [43]:
# Target Variables (y)
y_origin = train_df_origin['origin_risk_norm']
y_victim = train_df_victim['victim_risk_norm']

# Input Features (X)
X_origin_raw = train_df_origin[feature_cols]
X_origin = preprocessor.transform(X_origin_raw)
X_victim_raw = train_df_victim[feature_cols]
X_victim = preprocessor.transform(X_victim_raw)

In [44]:
# Train models
origin_model = GradientBoostingRegressor()
victim_model = GradientBoostingRegressor()

origin_model.fit(X_origin, y_origin)
victim_model.fit(X_victim, y_victim)

,loss,'squared_error'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [45]:
# Predict for all events
X_all = preprocessor.transform(events_df[feature_cols])

events_df['ml_origin_risk'] = origin_model.predict(X_all)
events_df['ml_target_risk'] = victim_model.predict(X_all)

In [46]:
events_df

,slug,original_method,event_date,reported_date,year,month,actor,actor_type,organization,industry_code,...,country,actor_country,state,county,change_log,apt_group,origin_risk_norm,victim_risk_norm,ml_origin_risk,ml_target_risk
0,1f72c2eb8ab303e4,1,2014-01-01,NaN,2014,1,Undetermined,Criminal,Barry University,61,...,United States of America,Undetermined,Florida,Miami-Dade,NaN,NaN,NaN,1.000000,0.473630,0.913313
1,ecac8b3e60a2f72f,1,2014-01-01,NaN,2014,1,Undetermined,Criminal,Record Assist LLC,54,...,United States of America,Undetermined,Texas,Harris,NaN,NaN,NaN,1.000000,0.550963,0.850001
2,3bbe0695e2d019f3,1,2014-01-01,NaN,2014,1,Syrian Electronic Army,Hacktivist,Skype's Social Media,54,...,United States of America,Syrian Arab Republic,Washington,King,NaN,NaN,0.538503,1.000000,0.510038,0.635988
3,6100014f6ca84b3d,1,2014-01-02,NaN,2014,1,Undetermined,Criminal,Snapchat,51,...,United States of America,Undetermined,California,Los Angeles,NaN,NaN,NaN,1.000000,0.593684,0.810195
4,3a94b8cf6dde1f66,1,2014-01-03,NaN,2014,1,DERP Trolling,Undetermined,Battle.net,51,...,United States of America,Undetermined,California,Orange,NaN,NaN,NaN,1.000000,0.519080,0.783511
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15784,g4h7k2p5d1r9m3t8,1,2025-03-14,2025-10-27,2025,3,Undetermined,Criminal,City of Gloversville (NY),92,...,United States of America,Undetermined,New York,Fulton,NaN,NaN,NaN,1.000000,0.831869,0.756021
15785,v8r3b1n6t4p9d2k7,1,2025-08-30,2025-10-27,2025,8,Undetermined,Criminal,Vibra Hospital of Sacramento,62,...,United States of America,Undetermined,California,Sacramento,NaN,NaN,NaN,1.000000,0.772193,0.917247
15786,m6r2t9p8d3a1k7f5,0,2025-05-01,2025-10-30,2025,5,Undetermined,Criminal,Undisclosed private company in Granada,54,...,Spain,Spain,NaN,NaN,NaN,NaN,0.357641,0.538054,0.658621,0.829289
15787,b6e2h9c4m7p1t3r8,1,2025-07-01,2025-10-29,2025,7,GRU Main Special Center for Special Technologi...,Nation-state,Undisclosed Ukrainian local government entity,92,...,Ukraine,Russian Federation,NaN,NaN,NaN,NaN,1.000000,0.679409,0.767900,0.617921
